# Multi-turn RL on MILES using Qwen3-1.7B on CodeContests

**Everything runs in this one environment.** The Megatron/SGLang trainer, the
Harbor agent server, and JupyterLab all run together here, and all services talk
over `localhost`.

**Modified Harbor (host execution).** This environment has no nested-container
runtime available, so the workshop uses a **modified version of Harbor** that
grades each task **directly on the host** (its `subprocess` environment, isolated
with `proot`) instead of spawning a separate container per task. From your point
of view nothing changes — Harbor still runs the agent and grades the solution —
it just executes on the host rather than in a per-task sandbox container.

Long-running services (Harbor, training) start in the background and are watched
via the log-tail / monitor cells. Paths come from the environment
(`REPO_DIR`, `EX`, `WORK_DIR`, `HARBOR_TASKS_DIR`); the CodeContests dataset is
already included.

## 0. Prerequisites

- Launched the workshop environment (this JupyterLab).
- An **idle GPU** (the workshop container exposes a single visible GPU).
- The CodeContests tasks + difficulty splits are **already included**
  (`$HARBOR_TASKS_DIR`, `$EX/data/cc_train_*.jsonl`); §5 below is a no-op
  unless you force a re-prep.

In [ ]:
import os, subprocess

# Paths come from the environment (preset when the workshop was launched).
REPO_DIR  = os.environ.get("REPO_DIR") or subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True, cwd=os.getcwd()).strip()
EX        = os.environ.get("EX",  os.path.join(REPO_DIR, "examples/experimental/qwen3-codecontests"))
RUNTIME   = os.path.join(EX, "runtime")
WORK_DIR  = os.environ.get("WORK_DIR",  os.path.join(RUNTIME, "work"))
TASKS_DIR = os.environ.get("HARBOR_TASKS_DIR", "/workspace/harbor_tasks_cc")
HF_CACHE  = os.environ.get("HF_HOME", os.path.join(RUNTIME, "cache/hf"))
WANDB_KEY = os.environ.get("WANDB_KEY", "")

os.environ.update(REPO_DIR=REPO_DIR, EX=EX, RUNTIME=RUNTIME, WORK_DIR=WORK_DIR,
                  HARBOR_TASKS_DIR=TASKS_DIR, HF_HOME=HF_CACHE, WANDB_KEY=WANDB_KEY)
for _d in (WORK_DIR, HF_CACHE):
    os.makedirs(_d, exist_ok=True)

for _k in ("REPO_DIR", "EX", "WORK_DIR", "HARBOR_TASKS_DIR", "HF_HOME"):
    print(f"{_k:16}= {os.environ[_k]}")
print(f"WANDB_KEY       = {'set' if WANDB_KEY else 'unset (W&B disabled)'}")
print(f"baked tasks     = {len(os.listdir(TASKS_DIR)) if os.path.isdir(TASKS_DIR) else 0}")


## 1. Clean slate

Clear any leftover trainer processes and ray/sglang session dirs from a previous
run, so this run starts clean.

In [ ]:
%%bash
# `ray stop --force` tears down the Ray head daemons (gcs_server, dashboard,
# monitor, log_monitor) that a crashed run leaves behind; pkill catches the
# job driver + any stragglers. (A plain pkill on 'raylet|sglang|train.py' misses
# the head daemons, whose command lines don't contain those words.)
ray stop --force >/dev/null 2>&1 || true
pkill -9 -f 'raylet|sglang|train\.py|run-qwen3-codecontests' 2>/dev/null || true
rm -rf /tmp/ray/session_* 2>/dev/null || true
# Clear STALE aiter JIT-compile locks. aiter guards each ROCm kernel build with a
# FileBaton lock file; if a run is killed mid-compile the lock is orphaned, and
# aiter's wait() loops FOREVER (no timeout) on the next run -> SGLang init wedges
# silently. Safe here: we just killed every compiler above, so no live baton.
rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true
live() { ps -eo stat,args --no-headers | grep -ivE '^Z| grep ' | grep -cE 'raylet|sglang|gcs_server|ray/dashboard|ray\.util|log_monitor|monitor\.py|train\.py|run-qwen3'; }
echo "live ray/sglang/train: $(live)"


## 2. Services run here

The trainer and the Harbor agent server both run in this environment, so there
is nothing separate to launch — continue to install miles and start Harbor.

## 3. Install miles (editable)

Miles source ships in the image but is installed editable here so local
edits to the bind-mounted repo take effect. (No-op cost if already installed.)

In [ ]:
%%bash
cd "$REPO_DIR" && pip install -e . --no-deps --no-build-isolation -q
python3 -c 'import miles, miles_plugins.mbridge' && echo 'miles import ok'


## 4. Start Harbor (background)

Launch the Harbor agent server in the background, then wait for
`localhost:11000/health`. It uses the modified **subprocess** environment
(`HARBOR_ENV_TYPE=subprocess`, preset) that grades tasks on the host via `proot`.

> **Where the output goes:** this cell starts Harbor *in the background* and
> redirects its output to `$WORK_DIR/harbor.log`, so the cell itself only prints
> a health line + a short log tail. To follow Harbor live:
> `tail -f $WORK_DIR/harbor.log`.

In [ ]:
%%bash
export MSWEA_CONFIG_FILE="$EX/harbor/codecontests.yaml" \
       HARBOR_TASKS_DIR="$HARBOR_TASKS_DIR" \
       HARBOR_TRIALS_DIR="${HARBOR_TRIALS_DIR:-$WORK_DIR/cc_trials}" \
       HARBOR_DELETE_CONTAINERS=true \
       OPENAI_API_KEY=dummy MSWEA_API_KEY=dummy
# (HARBOR_ENV_TYPE defaults to 'subprocess' in the image.)
LOG="$WORK_DIR/harbor.log"
if curl -sf http://localhost:11000/health >/dev/null 2>&1; then
  echo "Harbor already running on http://localhost:11000"
else
  # Detach the background server's stdio (setsid + </dev/null + >LOG) so it does
  # NOT hold this %%bash cell's output pipe open — otherwise the cell hangs and
  # shows nothing. The foreground health loop below then streams live.
  cd "$EX" && PYTHONPATH="$REPO_DIR" setsid "${HARBOR_PYTHON:-python3}" \
      harbor/server.py --port 11000 --max-concurrent 8 </dev/null >"$LOG" 2>&1 &
  echo "Starting Harbor (pid $!) — logging to: $LOG"
  for i in $(seq 1 30); do
    curl -sf http://localhost:11000/health >/dev/null 2>&1 && { echo "Harbor healthy on http://localhost:11000"; break; }
    [ "$i" = 30 ] && echo "Harbor did NOT become healthy in 60s — see the log tail below"
    sleep 2
  done
fi
echo "----- harbor.log (last 10 lines) -----"
tail -n 10 "$LOG" 2>/dev/null || echo "(no log yet)"


## 5. Data preparation (baked)

The CodeContests tasks and difficulty splits are **baked into the image**, so
this is normally a no-op. Run it only to rebuild from scratch (e.g. a
different dataset/curriculum); it downloads from HF and re-extracts.

In [ ]:
%%bash
if [ -d "$HARBOR_TASKS_DIR/code_contests-0000" ] && [ -f "$EX/data/cc_train_easy.jsonl" ]; then
  echo "data present (tasks=$(ls "$HARBOR_TASKS_DIR" | wc -l)); skipping prep"
else
  python3 "$EX/data_prep/extract_codecontests.py" --dataset open-thoughts/CodeContests --out "$HARBOR_TASKS_DIR"
  python3 "$EX/data_prep/split_by_difficulty.py" --tasks "$HARBOR_TASKS_DIR" --out-dir "$EX/data"
fi


### Task Directory structure
**What the agent must do.** Each task's `instruction.md` *is* the prompt. The agent (mini-swe-agent) is instructed to **write a Python solution to `/app/solution.py`** that reads from stdin and writes to stdout, test it on the sample, then submit by issuing `echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT`. The grader (`tests/test.sh`) runs `python3 /app/solution.py` against hidden tests with `pytest`; all pass → `reward.txt = 1`, otherwise `0`. (The modified Harbor runs this on the host, not in a per-task container.)

**Task directory** — `$TASKS_DIR/<task_id>/` (what Harbor serves):
```
code_contests-12701/
├── instruction.md          # the prompt (problem + I/O spec + "write /app/solution.py")
├── environment/Dockerfile  # task env spec (ignored here — graded on the host)
├── task.toml               # task metadata
└── tests/
    ├── test.sh             # grader: runs solution.py, writes reward.txt (1/0)
    ├── test_state.py       # pytest assertions
    └── test_data.json      # hidden test cases
```

**Trial output** — after a training (§7) run, each attempt lands in `$WORK_DIR/cc_trials/<task_id>__<id>/`:
```
agent/mini-swe-agent.txt              # FULL turn-by-turn transcript (human-readable)
agent/mini-swe-agent.trajectory.json  # machine-readable messages
verifier/reward.txt                   # 1 (passed) or 0
result.json                           # trial outcome
```

Below is one such **passed (reward = 1)** trial, shown turn by turn — the prompt, the model's actions, and the environment's responses, each colored distinctly.

### Example: A passed trial's trajectory (`code_contests-13226`. Agent gets reward = 1)

Colors: <span style='color:#2563eb'><b>prompt</b></span> &nbsp; <span style='color:#16a34a'><b>LLM</b></span> &nbsp; <span style='color:#d97706'><b>agent / environment</b></span>

<pre style='white-space:pre-wrap;font-family:monospace;font-size:12px;line-height:1.5'><span style='color:#2563eb'>PROMPT  (instruction.md the model receives)
# 1204_A. BowWow and the Timetable   (Codeforces rating 1000, tag: math)
Trains depart at times 4^k for every integer k &gt;= 0 (1, 4, 16, 64, ...).
You arrive at time s, given in BINARY without leading zeros (0 &lt;= s &lt; 2^100).
Output how many trains departed strictly before s (how many k have 4^k &lt; s).
Examples:  100000000 -&gt; 4    101 -&gt; 2    10100 -&gt; 3
Task: write a Python 3 solution to /app/solution.py (read stdin, write stdout).
Grader runs `python3 /app/solution.py` on hidden tests; submit with
`echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT` once correct.</span>

<span style='color:#16a34a'>step 1 · LLM  — writes the program, but with NO redirect (so it only prints):
  echo &#x27;&lt;program&gt;&#x27;
import sys
def main():
    s = int(sys.stdin.read(), 2)
    count = 0; k = 0
    while True:
        if 4 ** k &lt; s:
            count += 1; k += 1
        else:
            break
    print(count)
if __name__ == &quot;__main__&quot;:
    main()</span>

<span style='color:#d97706'>step 1 · agent  -&gt; returncode 0   (printed to stdout; /app/solution.py NOT created)</span>

<span style='color:#16a34a'>step 2 · LLM (tries to prematurely run the solution):
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 2 · agent  -&gt; returncode 2
  python3: can&#x27;t open file &#x27;/app/solution.py&#x27;: [Errno 2] No such file or directory</span>

<span style='color:#16a34a'>step 3 · LLM  — retries the same run:
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 3 · agent  -&gt; returncode 2
  python3: can&#x27;t open file &#x27;/app/solution.py&#x27;: [Errno 2] No such file or directory</span>

<span style='color:#16a34a'>step 4 · LLM  — now WITH a redirect, so the file is written:
  echo &#x27;&lt;same program&gt;&#x27; &gt; /app/solution.py</span>

<span style='color:#d97706'>step 4 · agent  -&gt; returncode 0   (empty output; file written)</span>

<span style='color:#16a34a'>step 5 · LLM:
  python3 /app/solution.py</span>

<span style='color:#d97706'>step 5 · agent  -&gt; returncode 1
  ValueError: invalid literal for int() with base 2: &#x27;&#x27;   (nothing piped to stdin in this bare run)</span>

<span style='color:#16a34a'>step 6 · LLM  — submit:
  echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT</span>

<span style='color:#d97706'>step 6 · agent  -&gt; action was not executed   (mini-swe-agent submit sentinel; episode ends)</span>

result -&gt; reward = 1   (the grader pipes each hidden test into the saved /app/solution.py, which is correct, so pytest passes)</pre>

## 6. Model

Pre-fetch **`Qwen/Qwen3-1.7B`** into the HF cache (`$HF_CACHE`, default `runtime/cache/hf`, via `HF_HOME`) with `snapshot_download`, so the weights are local before training.

**Optional:** the first rollout/training run (next section) calls the launcher **without** `--skip-prepare`, whose `prepare()` step already downloads `Qwen/Qwen3-1.7B` and then converts it to a Megatron `torch_dist` checkpoint. This cell just separates the ~3.4 GB download from that first run — skip it if you don't mind the first run doing the download itself.

In [ ]:
%%bash
echo "Fetching Qwen/Qwen3-1.7B into the HF cache (HF_HOME=$HF_HOME)..."
python3 -c 'from huggingface_hub import snapshot_download; print(snapshot_download("Qwen/Qwen3-1.7B"))'
echo "weights ready (cache size: $(du -sh "$HF_HOME" 2>/dev/null | cut -f1))"


## 7. Training run

The GRPO loop (rollout → reward → update → weight-sync → checkpoint), run in
THIS container. Rollouts are graded by the Harbor server on `localhost:11000`
via the subprocess environment. Launched detached; watch the monitor cell.

**One run at a time** — if an earlier run is live, reset first (§9).

> **Where the output goes:** training runs *in the background*; all of its output
> goes to `$WORK_DIR/train.log` (the launch cell stays quiet by design so you can
> keep using the notebook). Use the **monitor cell (§8)** to watch progress, or
> `tail -f $WORK_DIR/train.log` in a terminal.


In [ ]:
%%bash
cd "$EX"
# MILES_HOST_IP=127.0.0.1 is load-bearing: miles derives the SGLang router / TITO
# session-server bind address AND the agent's OPENAI_API_BASE from get_host_info(),
# which otherwise probes the LAN IP (e.g. 172.17.0.2). The agent runs in a proot
# sandbox and dials localhost; without pinning loopback every model call fails
# with a litellm Connection error -> AgentError -> reward 0.
export WANDB_KEY="$WANDB_KEY" \
       MILES_HOST_IP="${MILES_HOST_IP:-127.0.0.1}" \
       MILES_ROUTER_EXTERNAL_HOST="${MILES_ROUTER_EXTERNAL_HOST:-127.0.0.1}" \
       AGENT_SERVER_URL=http://localhost:11000 \
       HARBOR_TASKS_DIR="$HARBOR_TASKS_DIR" \
       WANDB_DIR="$WORK_DIR/wandb"
LOG="$WORK_DIR/train.log"
# Belt-and-suspenders: clear a STALE aiter JIT lock from a previously interrupted
# run BEFORE launching, but only if nothing is compiling right now (so we never
# yank a baton from a live compile). A stale lock makes SGLang init hang forever.
if ! pgrep -f 'sglang::|run-qwen3-codecontests|train\.py' >/dev/null 2>&1; then
  rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true
fi
# Detach stdio (setsid + </dev/null + >LOG) so the background trainer doesn't
# hold this cell's output pipe open (which would hang the cell with no output).
PYTHONPATH="$REPO_DIR" setsid python3 run-qwen3-codecontests.py \
    --prompt-data "$EX/data/cc_train_easy.jsonl" \
    --num-rollout 20 \
    --rollout-batch-size 2 \
    --n-samples-per-prompt 8 \
    --global-batch-size 16 \
    --max-seq-len 16384 --save-interval 2 </dev/null >"$LOG" 2>&1 &
echo "Training launched in the background (pid $!)."
echo "  Logs:        $LOG"
echo "  Watch live:  run the monitor cell (below), or in a terminal:  tail -f $LOG"
echo "  Startup takes a few minutes (Megatron init + SGLang engine load) before the first rollout."


## 8. Monitoring Training

One cell drives the whole run. Every refresh it prints, **pinned at the top so it never scrolls away**, the **W&B run link** (parsed from `train.log`) followed by a **live `train.log` tail**. Then:

- **During the rollout phase** (before the first training step finishes) it **reuses the rollout monitor** (`rollout_panel` from `notebook-monitoring/`) to show the per-sample turns/reward view.
- **After the first full rollout + training step**, it switches to the live, **hover-able** line charts below.

Live, **hover-able** line charts rendered **in the notebook** (so a non-W&B viewer sees the same curves), parsed from `train.log`:

- **Timing** — `step_time` (`perf/step_time`) vs `rollout_time` (`perf/rollout_time`).
- **Reward** — `rollout/raw_reward` (pass rate) per step.
- **Truncated** — `rollout/truncated` per step.

Hover any point to read its **Y value** (plotly; auto-installed on first run, `hovermode="x unified"` shows all series at that step). Below the charts: a per-step table (re-printed every refresh, so every step is shown) and the **turns-per-sample** for the latest rollout. Re-renders in place; **interrupt (stop) to end** — this only reads logs and does not affect training. These mirror the W&B `perf/*`, `rollout/*` series.

In [22]:
# Live training monitor: W&B link + train.log tail + current-step rollout panel +
# hover-able charts. Definitions live in notebook-monitoring/training_monitor.py. Interrupt to end.
import os, sys, importlib
sys.path.insert(0, os.path.join(
    os.environ.get("EX", os.path.join(os.getcwd(), "examples/experimental/qwen3-codecontests")),
    "notebook-monitoring"))
import rollout_monitor, training_monitor
importlib.reload(rollout_monitor); importlib.reload(training_monitor)
training_monitor.watch_training()


W&B run: (link not in train.log yet)

=== train.log (tail) ===
(waiting for log...)

=== step 4 rollout: 2 problems | 16 samples | submitted 8 | running 1 | other 7 | reward>0: 1 ===
-- Submitted (8) --
[2] code_contests-8447  reward=0.0  turns=11
     cmd: python3 /app/solution.py
     out: <returncode>2</returncode> <output> python3: can't open file '/app/solution.py': [Errno 2] No such file or directory </o
[3] code_contests-8447  reward=0.0  turns=3
     cmd: echo '4'
     out: <returncode>0</returncode> <output> 4 </output>
[4] code_contests-8447  reward=0.0  turns=5
     cmd: echo 2
     out: <returncode>0</returncode> <output> 2 </output>
[8] code_contests-8447  reward=0.0  turns=7
     cmd: python3 /app/solution.py
     out: <returncode>2</returncode> <output> python3: can't open file '/app/solution.py': [Errno 2] No such file or directory </o
[9] code_contests-8447  reward=0.0  turns=13
     cmd: echo 'import sys def main(): data = sys.stdin.read().split() idx = 0 t = int(data

step  step_t  roll_t  reward  trunc
   0     519     383   0.250  0.438
   1     521     501   0.250  0.062
   2     734     643   0.188  0.250
   3     545     514   0.062  0.438
step_t = perf/step_time reported by the trainer for each step (its own measurement, including step 0); not wall-clock timed by this monitor.

[stopped]


## 9. Stop / reset between runs

**Run only ONE rollout/training at a time.** Reset by killing the
ray/sglang/train processes and clearing their session dirs. If endpoint errors
persist after a reset, relaunch the whole workshop environment. Re-run §4 if
Harbor was killed.

In [ ]:
%%bash
# Full teardown between runs: stop the Ray cluster (head daemons included), then
# kill the job driver / trainer and clear session + JIT lock files.
ray stop --force >/dev/null 2>&1 || true
pkill -9 -f 'raylet|sglang|train\.py|run-qwen3-codecontests' 2>/dev/null || true
sleep 3
rm -rf /tmp/ray/session_* 2>/dev/null || true
rm -f /app/aiter/aiter/jit/build/lock_* 2>/dev/null || true
live() { ps -eo stat,args --no-headers | grep -ivE '^Z| grep ' | grep -cE 'raylet|sglang|gcs_server|ray/dashboard|ray\.util|log_monitor|monitor\.py|train\.py|run-qwen3'; }
echo "after reset; live ray/sglang/train: $(live)"
echo 'NOTE: Harbor was NOT killed (different process); re-run the Start Harbor cell only if you killed it.'
